<h1><a id="title"></a>Uncertainty from a difference raster</h1>

Start directly from a **vertical difference GeoTIFF** (skip point-cloud/DEM
differencing) and propagate uncertainty over standardized areas of interest.

Two modes, chosen with <code>TIF_IS_BIAS_REMOVED</code>:

<ul>
<li><b>Raw difference</b> (<code>False</code>): interactively pick stable areas &rarr;
remove the median bias &rarr; fit the variogram &rarr; per-feature + AOI uncertainty.</li>
<li><b>Already bias-removed</b> (<code>True</code>): the TIF is already stable-masked and
de-biased &rarr; fit the variogram &rarr; AOI uncertainty (no stable-area step).</li>
</ul>

The difference raster must be in a projected CRS in meters (e.g. UTM).

<h2><a id="setup"></a>Setup</h2>

In [ ]:
import os, sys, pathlib

# --- Colab guard ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1) Mount Drive (idempotent)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # 2) Check if condacolab is already fully configured
    # We verify both that conda exists AND that LD_LIBRARY_PATH is set
    conda_ready = (
        os.path.exists("/usr/local/bin/conda") and
        "LD_LIBRARY_PATH" in os.environ
    )
    
    if conda_ready:
        import condacolab
        condacolab.check()
        print("âœ“ Condacolab already installed and configured")
    else:
        print("Installing condacolab (kernel will restart)...")
        !pip install -q condacolab
        import condacolab
        condacolab.install()  # This restarts the kernel
else:
    print("Not running in Colab; skipping condacolab setup.")

**Kernel Restart Required (Colab):** if condacolab was just installed, the kernel
restarts automatically. After it restarts, continue from the next cell.

In [ ]:
import os, sys, pathlib
from pathlib import Path

if IN_COLAB:
    # Verify condacolab and install PDAL
    import condacolab
    condacolab.check()
    
    # Remove conflicting Python pin and install PDAL
    !rm -f /usr/local/conda-meta/pinned
    !mamba install -y -c conda-forge pdal python-pdal
    
    # Fix SQLite symlink - conda installs newer SQLite but old symlink remains
    # Dynamically find the installed SQLite version instead of hardcoding
    !sqlite_lib=$(ls /usr/local/lib/libsqlite3.so.3.* 2>/dev/null | head -1) && \
        if [ -n "$sqlite_lib" ]; then \
            sudo ln -sf "$sqlite_lib" /usr/local/lib/libsqlite3.so.0; \
            echo "Linked $sqlite_lib -> libsqlite3.so.0"; \
        fi
    
    # Fix numpy version conflict
    !{sys.executable} -m pip install -q "numpy<2.2"
    
    # Set PROJ environment
    os.environ['PROJ_LIB'] = '/usr/local/share/proj/'
    
    # Install topochange from GitHub
    print("\nInstalling topochange package...")
    !{sys.executable} -m pip install -q --no-cache-dir git+https://github.com/Cassandra-Brigham/topochange.git
    
    # Install additional packages not in topochange dependencies
    !{sys.executable} -m pip install -q small-gicp colormaps boto3
    
    # Verify PDAL via wrapper
    from topochange.pdal_wrapper import pdal, get_pdal_status
    status = get_pdal_status()
    print(f"\nEnvironment ready!")
    print(f"  PDAL version: {status['version']}")
    print(f"  PDAL mode: {status['mode']}")
else:
    print("Not running in Colab; skipping environment setup.")

In [ ]:
# Install visualization libraries
%pip install -q comm ipywidgets
%pip install -q ipyleaflet

# Fix pyproj PROJ path (Colab-specific, may need adjustment based on conda installation)
import os
import sys

# Set PROJ_LIB if needed (conda usually handles this automatically)
if IN_COLAB and not os.environ.get("PROJ_LIB"):
    from google.colab import output
    output.enable_custom_widget_manager()
    # Try common conda locations first
    possible_proj_paths = [
        "/opt/conda/share/proj",
        "/usr/share/proj", 
        "/usr/local/share/proj"
    ]
    for proj_path in possible_proj_paths:
        if os.path.isdir(proj_path):
            os.environ["PROJ_LIB"] = proj_path
            break

<h3><a id="import-libraries"></a>Import required libraries</h3>

In [ ]:
import numpy as np
import rasterio

# Core classes
from topochange import (
    Raster,
    RasterPair,
    RasterDataHandler,
    StatisticalAnalysis,
    SingleVariogram,
    GridVariogram,
    RegionalUncertaintyEstimator,    
)

# Interactive features (requires ipyleaflet)
from topochange import (
    TopoMapInteractor,
    StableAreaRasterizer,
    StableAreaAnalyzer,
)

# Geoid utilities (used for Colab PROJ grid setup)
from topochange.geoid_utils import (
    ensure_proj_grids_for_region,
    get_all_proj_data_dirs,
)

SEED = 42

# Ensure PROJ geoid grids are available (Colab only)
if IN_COLAB:
    print("Setting up PROJ geoid grids for Colab...")
    print(f"PROJ data directories: {get_all_proj_data_dirs()}")
    ensure_proj_grids_for_region('us_noaa', verbose=True)
    print("\nPROJ geoid grids ready")

<h2><a id="config"></a>Configuration</h2>

Set the path to your difference raster and choose the mode.

In [ ]:
from pathlib import Path


DIFF_TIF_PATH = "your/path/to/difference.tif"

# Is DIFF_TIF_PATH ALREADY the stable-masked, bias-removed difference raster?
#   False -> raw difference: stable-area selection -> bias removal -> variogram
#   True  -> already bias-removed: skip straight to variogram + uncertainty
TIF_IS_BIAS_REMOVED = False

# Raw-mode only: basemap raster for the interactive stable-area selector.
# Provide a hillshade (preferred) or a DEM. If it is a DEM (not a hillshade),
# set HILLSHADE_IS_DEM = True to auto-generate a hillshade from it. If left
# None, the difference raster itself is used as the basemap.
HILLSHADE_TIF_PATH = None
HILLSHADE_IS_DEM = False

# Output directory for polygons, shapefiles, and results.
if not Path(DIFF_TIF_PATH).exists():
    raise FileNotFoundError(
        f"DIFF_TIF_PATH not found: {DIFF_TIF_PATH}\n"
        "Set DIFF_TIF_PATH to your vertical difference GeoTIFF."
    )
BASE_DATA_DIR = Path(DIFF_TIF_PATH).resolve().parent
BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)

UNIT = "m"

print(f"Difference raster : {DIFF_TIF_PATH}")
print(f"Mode              : {'already bias-removed' if TIF_IS_BIAS_REMOVED else 'raw difference (full analysis)'}")
print(f"Outputs           : {BASE_DATA_DIR}")

<h2><a id="load"></a>Load the difference raster</h2>

In [ ]:
import rasterio
from IPython.display import display

# The difference raster is the analysis surface (AOIs are placed on its valid data)
diff = Raster.from_file(DIFF_TIF_PATH)
with rasterio.open(DIFF_TIF_PATH) as _src:
    DEM_RESOLUTION = float(abs(_src.res[0]))
    _crs = _src.crs
print(f"Loaded difference raster: {diff.filename}")
print(f"  resolution: {DEM_RESOLUTION} m | CRS: {_crs}")

# Basemap for the stable-area selector (raw mode only)
hillshade1 = None
if not TIF_IS_BIAS_REMOVED:
    if HILLSHADE_TIF_PATH:
        _base = Raster.from_file(HILLSHADE_TIF_PATH)
        hillshade1 = _base.hillshade() if HILLSHADE_IS_DEM else _base
    else:
        print("No HILLSHADE_TIF_PATH set -> using the difference raster as the basemap.")
        hillshade1 = diff

<h2><a id="error_analysis"></a>Error analysis</h2>
<h3><a id="define_stable_areas"></a>Define stable areas (raw mode)</h3>

Draw stable polygons (and, for per-feature uncertainty, unstable features) on the
map. Skipped automatically when <code>TIF_IS_BIAS_REMOVED = True</code>.

In [ ]:
# Draw stable (and unstable) polygons on the map. Skipped if already bias-removed.
if not TIF_IS_BIAS_REMOVED:
    import os
    out_folder_poly = Path.joinpath(BASE_DATA_DIR, "polygons/")
    os.makedirs(out_folder_poly, exist_ok=True)
    interactor = TopoMapInteractor(
        topo_diff_path=diff.filename,
        hillshade_path=hillshade1.filename,
        output_dir=out_folder_poly,
        overlay_dpi=600,
        overlay_vmin=-10,
        overlay_vmax=10,
    )
    display(interactor.map)
else:
    print("TIF_IS_BIAS_REMOVED=True -> skipping interactive stable-area selection.")

<h3><a id="estimate-error"></a>Estimate systematic error</h3>

Build <code>raster_bias_removed</code> &mdash; by removing the median bias over the drawn
stable areas (raw mode), or straight from the provided TIF (bias-removed mode).

In [ ]:
# Produce `raster_bias_removed` (a RasterDataHandler) for the variogram.
if not TIF_IS_BIAS_REMOVED:
    stable_polys, _ = interactor.export_geodataframes()

    # rasterize the stable polygons into a combined stable-area mask
    rasterizer_stable = StableAreaRasterizer(interactor.topo_diff.path, stable_polys, nodata=-9999)
    analyzer_stable = StableAreaAnalyzer(rasterizer_stable)
    stable_area_path = Path.joinpath(BASE_DATA_DIR, "polygons/combined_stable.tif")
    df_all_stable_polys = analyzer_stable.stats_all(stable_area_path)

    # remove the median vertical bias measured over the stable areas
    rdh = RasterDataHandler(str(stable_area_path), UNIT, DEM_RESOLUTION)
    rdh.load_raster()
    vertical_bias = float(np.median(rdh.data_array))
    bias_removed_path = Path.joinpath(BASE_DATA_DIR, "polygons/combined_stable_bias_removed.tif")
    rdh.subtract_value_from_raster(bias_removed_path, vertical_bias)

    raster_bias_removed = RasterDataHandler(str(bias_removed_path), UNIT, DEM_RESOLUTION)
    raster_bias_removed.load_raster()
    print(f"Median vertical bias removed: {vertical_bias:.4f} m")
else:
    raster_bias_removed = RasterDataHandler(str(DIFF_TIF_PATH), UNIT, DEM_RESOLUTION)
    raster_bias_removed.load_raster()
    print("Using the provided TIF directly as the bias-removed raster.")

print(f"Bias-removed stats: mean={np.mean(raster_bias_removed.data_array):.4f}, "
      f"median={np.median(raster_bias_removed.data_array):.4f}, "
      f"std={np.std(raster_bias_removed.data_array):.4f}")

<h3><a id="Variography"></a>Variography</h3>

In [ ]:
sv = SingleVariogram(
    raster_bias_removed
)

sv.compute_empirical_variogram(
    area_side=250,
    samples_per_area=400,
    max_samples=10_000_000,
    bin_width=10,
    max_lag_multiplier=1/3,
    estimator="cressie_hawkins",
    return_sample=True,   
)

sv.fit_model(
    model_types=['spherical','matern','exponential'],
    include_nugget = True,
    max_components=3,
    criterion="aicc",
)

sv.plot_single_variogram(
    include_model = True,
)

In [ ]:
gv = GridVariogram(raster_bias_removed, n_realizations=1)
gv.run(
    area_side=250, samples_per_area=400, max_samples=10_000_000,
    bin_width=10, max_lag_multiplier=1/3,
    estimator="matheron", n_samples=10,
    criterion="aicc", seed=42, verbose=True,
)

In [ ]:
gv.plot_variogram(include_central_model=True,include_bootstrap=True)

<h2><a id="uncertainty_propagation"></a>Uncertainty propagation</h2>
<h3>Hand-drawn features (raw mode)</h3>

In [ ]:
# Uncertainty for each hand-drawn feature of interest (raw mode only).
from shapely.ops import unary_union

if not TIF_IS_BIAS_REMOVED:
    _, unstable_polys = interactor.export_geodataframes()
    # stable terrain = raster footprint minus ALL drawn features
    _all_features = unary_union(list(unstable_polys['geometry']))
    uncertainties_per_feature = []
    for i, poly in enumerate(unstable_polys['geometry']):
        print(f"Processing feature {i + 1}/{len(unstable_polys)}...")
        estimator = RegionalUncertaintyEstimator(
            raster_data_handler=raster_bias_removed,
            variogram_analysis=gv,
            area_of_interest=poly,
            unstable_geoms=_all_features,  # -> calibrated median (bias) SE
            fitted_model=gv.fitted_model,
        )
        estimator.calc_total_uncertainty(n_pairs=25_000, seed=SEED)
        print(f"  median (bias) SE: {estimator.sigma_a_stable:.4f} m"
              f"  [{estimator.stable_geom_source}]")
        uncertainties_per_feature.append(estimator)
    if uncertainties_per_feature:
        print("\n" + uncertainties_per_feature[0].summary())
else:
    print("Bias-removed mode: no hand-drawn features. See the standardized AOI section below.")
    
## figure out bias removed path...

<h2><a id="export"></a>Export results</h2>

Save the fitted variogram model, the uncertainty tables and summaries, and the figures
to a <code>results/</code> sub-folder (figures under <code>results/figures/</code>).

In [ ]:
# --- Export results: variogram + uncertainty summaries, tables, and figures ---
import json
import matplotlib.pyplot as plt

# Everything lands in a `results/` sub-folder (figures under results/figures/).
results_dir = Path.joinpath(BASE_DATA_DIR, "results")
figures_dir = Path.joinpath(results_dir, "figures")
figures_dir.mkdir(parents=True, exist_ok=True)
written = []

# Save figures as Illustrator-editable vectors (real text, vector shapes -- not
# outlined/Type-3 text or a flat raster).
plt.rcParams["pdf.fonttype"] = 42   # embed TrueType so text stays selectable/editable
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"  # keep SVG text as <text>, not outlined paths


def _save_fig(fig, name, formats=("pdf", "svg", "png")):
    # PDF/SVG are the editable vector versions; PNG is a raster preview.
    # dpi only affects rasterized layers (e.g. the difference imshow); all
    # axes, lines, markers, and text stay vector and editable.
    stem = Path(name).stem
    for ext in formats:
        p = Path.joinpath(figures_dir, f"{stem}.{ext}")
        fig.savefig(p, dpi=300, bbox_inches="tight")
        written.append(p)


# ---- Variogram information ----
try:
    with open(Path.joinpath(results_dir, "variogram_summary.txt"), "w") as f:
        f.write(gv.summary())
    written.append(Path.joinpath(results_dir, "variogram_summary.txt"))
except Exception as e:
    print(f"  (variogram summary skipped: {e})")

try:
    _fm = gv.fitted_model
    _sill = _fm.composite_model.get_total_sill()
    _has_boot = _fm.param_samples is not None and len(_fm.param_samples) > 0
    vg = {
        "central_model": getattr(gv, "central_model_name", None),
        "total_sill": float(_sill) if _sill is not None else None,
        "has_bootstrap_samples": bool(_has_boot),
        "n_bootstrap_samples": int(len(_fm.param_samples)) if _fm.param_samples is not None else 0,
        "central_params": getattr(gv, "central_params", None),
    }
    with open(Path.joinpath(results_dir, "variogram_model.json"), "w") as f:
        json.dump(vg, f, indent=2, default=str)
    written.append(Path.joinpath(results_dir, "variogram_model.json"))
except Exception as e:
    print(f"  (variogram model json skipped: {e})")

try:
    _r = gv.plot_variogram(include_central_model=True, include_bootstrap=True)
    if isinstance(_r, tuple):
        _r = _r[0]
    _fig = _r if hasattr(_r, "savefig") else plt.gcf()
    _save_fig(_fig, "variogram")
except Exception as e:
    print(f"  (variogram figure skipped: {e})")

# ---- Statistical distribution figure (histogram of differences) ----
try:
    if "stats" in dir():
        _sr = stats.plot_data_stats()
        _sfig = _sr if hasattr(_sr, "savefig") else plt.gcf()
        _save_fig(_sfig, "distribution")
    else:
        print("  (distribution figure skipped: no `stats` in this notebook)")
except Exception as e:
    print(f"  (distribution figure skipped: {e})")

# ---- Uncertainty information ----
try:
    aoi_results.to_csv(Path.joinpath(results_dir, "aoi_uncertainty_results.csv"), index=False)
    written.append(Path.joinpath(results_dir, "aoi_uncertainty_results.csv"))
except Exception as e:
    print(f"  (aoi results csv skipped: {e})")

try:
    with open(Path.joinpath(results_dir, "aoi_uncertainty_summaries.txt"), "w") as f:
        for (nm, _, _), est in zip(aoi_polys, aoi_estimators):
            f.write(f"===== {nm} =====\n{est.summary()}\n\n")
    written.append(Path.joinpath(results_dir, "aoi_uncertainty_summaries.txt"))
except Exception as e:
    print(f"  (aoi summaries skipped: {e})")

# Hand-drawn feature summaries (only if the per-feature loop ran)
if "uncertainties_per_feature" in dir() and uncertainties_per_feature:
    try:
        with open(Path.joinpath(results_dir, "feature_uncertainty_summaries.txt"), "w") as f:
            for i, est in enumerate(uncertainties_per_feature):
                f.write(f"===== feature {i + 1} =====\n{est.summary()}\n\n")
        written.append(Path.joinpath(results_dir, "feature_uncertainty_summaries.txt"))
    except Exception as e:
        print(f"  (feature summaries skipped: {e})")

# ---- AOI placement figure (difference raster + AOI outlines) ----
try:
    with rasterio.open(diff.filename) as ds:
        arr = ds.read(1, masked=True)
        extent = [ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top]
    _vals = np.asarray(arr.compressed(), dtype=float)
    _vals = _vals[np.isfinite(_vals)]
    _v = float(np.percentile(np.abs(_vals), 98)) if _vals.size else 1.0
    _v = _v if _v > 0 else 1.0
    figA, axA = plt.subplots(figsize=(9, 9))
    imA = axA.imshow(arr, extent=extent, origin="upper", cmap="RdBu", vmin=-_v, vmax=_v, rasterized=True)
    for nm, poly, _ in aoi_polys:
        xs, ys = poly.exterior.xy
        axA.plot(xs, ys, lw=1.6, label=nm)
    axA.set_aspect("equal")
    axA.set_title("Standardized AOI polygons on the difference raster")
    axA.legend(fontsize=7, loc="upper right")
    figA.colorbar(imA, ax=axA, shrink=0.7, label="Elevation difference (m)")
    _save_fig(figA, "aoi_polygons")
except Exception as e:
    print(f"  (aoi figure skipped: {e})")

print(f"Exported {len(written)} item(s) to {results_dir}:")
for w in written:
    print("  -", Path(w).relative_to(results_dir))
